In [1]:
%pip install -q transformers

from transformers import pipeline
import pandas as pd


pipeline = pipeline(model="mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")

data = pd.read_csv("analyst_ratings_processed.csv")
print(data.columns)

Note: you may need to restart the kernel to use updated packages.


Device set to use cuda:0


Index(['Unnamed: 0', 'title', 'date', 'stock'], dtype='object')


In [58]:
#Blackberrys ticker is BB but in the database it hasd it as BBRY
stock = "BBRY"

stock = data[data['stock'] == stock].copy()

headline = stock['title'].dropna().tolist()

print(len(headline))

2570


In [59]:
results = []
batch = 32

#Run in batch when using GPU to increase speeds
for i in range(0, len(headline), batch):
    test = headline[i:i+batch]
    results.extend(pipeline(test))
print(results)

[{'label': 'positive', 'score': 0.9996532201766968}, {'label': 'positive', 'score': 0.999618649482727}, {'label': 'neutral', 'score': 0.9790917038917542}, {'label': 'neutral', 'score': 0.9998241066932678}, {'label': 'negative', 'score': 0.9984152317047119}, {'label': 'neutral', 'score': 0.9998809099197388}, {'label': 'neutral', 'score': 0.9998612403869629}, {'label': 'neutral', 'score': 0.9997915625572205}, {'label': 'neutral', 'score': 0.980026364326477}, {'label': 'neutral', 'score': 0.9998316764831543}, {'label': 'neutral', 'score': 0.9998316764831543}, {'label': 'positive', 'score': 0.9996358156204224}, {'label': 'negative', 'score': 0.9988073110580444}, {'label': 'neutral', 'score': 0.9998798370361328}, {'label': 'neutral', 'score': 0.9997579455375671}, {'label': 'neutral', 'score': 0.9790917038917542}, {'label': 'positive', 'score': 0.9048817753791809}, {'label': 'neutral', 'score': 0.9998227953910828}, {'label': 'positive', 'score': 0.6533893942832947}, {'label': 'positive', 'sc

In [60]:
stock = stock.dropna(subset=["title"]).copy()

stock['sentiment'] = [r['label'] for r in results]

score = {"neutral" : 0, "negative": -1, "positive": 1}

stock['sentiment'] = stock['sentiment'].str.lower().map(score)

stock['date'] = pd.to_datetime(stock['date'], utc=True).dt.tz_localize(None).dt.normalize()

print(stock['date'])

138467   2020-06-08
138468   2020-06-05
138469   2020-06-03
138470   2020-06-02
138471   2020-06-02
            ...    
141034   2013-01-30
141035   2013-01-30
141036   2013-01-30
141037   2012-03-06
141038   2011-12-01
Name: date, Length: 2570, dtype: datetime64[ns]


In [61]:
stock = stock.groupby('date')['sentiment'].sum().reset_index()

print(stock[stock['sentiment'] > 1])

          date  sentiment
5   2013-02-04          4
17  2013-02-25          3
18  2013-02-26          2
24  2013-03-07          2
26  2013-03-11          4
..         ...        ...
733 2018-09-28          2
750 2018-12-20          2
760 2019-03-29          2
771 2019-06-26          2
802 2019-12-20          3

[80 rows x 2 columns]


In [62]:
stock.to_csv('BB_Sentiment.csv', index=False)